In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

In [4]:
df_feat = pd.read_csv("text_representation/TF_IDF/text_representation_cleaned_tfidf.xls")
df_labels = pd.read_csv("schemas/fully_cleaned.csv")
# Merge features and labels
mapping = {
"negative": -1,
"neutral": 0,
"positive": 1
}
df_feat["sentiment"] = df_labels["sentiment"]
df_feat["sentiment"] = df_feat["sentiment"].map(mapping)

X = df_feat.drop("sentiment", axis=1)
y = df_feat["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------- Linear Regression --------
ln_reg = LinearRegression(fit_intercept=True, positive=False)  # Use best params from GridSearchCV
ln_reg.fit(X, y)
ln_pred = ln_reg.predict(X)
def convert_to_class(x):
    if x > 0.5:
        return "positive"
    elif x < -0.5:
        return "negative"
    else:
        return "neutral"

ln_pred_class = [convert_to_class(x) for x in ln_pred]
y_test_class = [convert_to_class(x) for x in y]
X["sentiment"] = y_test_class
X["ln_pred"] = ln_pred_class

print(classification_report(y_test_class, ln_pred_class))


print("TF-IDF Cleaned Linear regression F1 Score:,", f1_score(y_test_class, ln_pred_class, average='weighted'))
print("TF-IDF Cleaned Linear regression Precision:", precision_score(y_test_class, ln_pred_class, average='weighted'))
print("TF-IDF Cleaned Linear regression Recall:", recall_score(y_test_class, ln_pred_class, average='weighted'))
print("TF-IDF Cleaned Linear regression Accuracy Score:", accuracy_score(y_test_class, ln_pred_class))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         6
     neutral       1.00      1.00      1.00        11
    positive       1.00      1.00      1.00        33

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50

TF-IDF Cleaned Linear regression F1 Score:, 1.0
TF-IDF Cleaned Linear regression Precision: 1.0
TF-IDF Cleaned Linear regression Recall: 1.0
TF-IDF Cleaned Linear regression Accuracy Score: 1.0


# Error Analysis Theory:
Since we see in the above cell output that our labels is mostly unbalanced as it has about 66% positive labels, the final model will have difficulties addressing the predictions and will most probably predict the majority as positive reviews.